# Extruded-spline surface over the activation PLS projection

Fits a curve through the PLS point cloud, extrudes it along PLS3 into a surface,
re-expresses every point in the surface coordinates `(t, u)`, and predicts the
ordered time-horizon class from them.

The projection is the layer 18 + layer 27 sweep at position -1 (`pls6`), fitted
in Colab and written out as one CSV per dataset plus a shared `.npz` of the PLS
parameters. That replaces the earlier single-layer `layer_out-21` joblib
projection; the modelling below is unchanged, only the point cloud is new.

## Setup


In [ ]:
# Reload edited modules automatically: the helpers under `vendor/utils/` change
# often, and without this an already-imported module keeps its stale copy for the
# life of the kernel (an ImportError for a function that plainly exists on disk).
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import plotly.express as px
from pprint import pprint

## Models

In [ ]:
import sys
from pathlib import Path

# The saved spline/surface artifacts unpickle classes from `temporal_manifolds`;
# a minimal copy of that code lives in `vendor/` at the repo root.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vendor").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "vendor") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "vendor"))

# The layer-18 PLS model is a plain `.npz` of fitted arrays rather than a pickled
# sklearn estimator, so there is no `load_pls_model` here -- the arrays are the
# model. Nothing downstream needs to re-project raw activations (the CSV already
# carries the components), so this cell only reports what was fitted.
PLS_MODEL_PATH = REPO_ROOT / "models" / "layer18_27_pos-1_pls6_model.npz"

pls = dict(np.load(PLS_MODEL_PATH, allow_pickle=False))
pls_metadata = {
    "layers": pls["layer"].tolist(),
    "position": int(pls["position"][0]),
    "component_count": int(pls["components"][0]),
    "feature_count": int(pls["x_mean"].shape[0]),
    "residual_components": int(pls["residual_components"][0]),
    "residual_pca_explained_variance_ratio": pls["residual_pc_explained_variance_ratio"].tolist(),
}
pprint(pls_metadata)


## Data and horizon classes

In [ ]:
# Load the projection and derive the coarse temporal-scale classes.
# Class i covers [edge_i, edge_{i+1}), with edges at 1 second, 1 minute, 1 hour,
# 1 day, 1 week, 1 month, 1 year, 1 decade, 1 century, +inf. Month = 30.4375 days,
# matching `temporal_manifolds.horizon.cache.UNIT_TO_MONTHS` in
# temporal-manifolds-last-position.
from utils.horizon_classes import (
    HORIZON_CLASS_LABELS,
    UNIT_TO_MONTHS,
    horizon_class,
    horizon_class_label,
)

# The Colab export names its columns `pls_1..pls_6` / `residual_pc_1..3`; the rest
# of this notebook (and the saved artifacts) use the older `PLS1..PLS3` /
# `reconstruction_residual_PC1..3` names, so rename on the way in. Only the first
# three PLS components enter the surface fit, as before; PLS4-6 are kept for the baseline cell at the end.
DATA_PATH = REPO_ROOT / "data" / "all_datasets_layer18_27_pos-1_pls6.csv"

COLUMN_RENAMES = {
    "pls_1": "PLS1",
    "pls_2": "PLS2",
    "pls_3": "PLS3",
    "pls_4": "PLS4",
    "pls_5": "PLS5",
    "pls_6": "PLS6",
    "residual_pc_1": "reconstruction_residual_PC1",
    "residual_pc_2": "reconstruction_residual_PC2",
    "residual_pc_3": "reconstruction_residual_PC3",
    "dataset": "source_folder",
}

# This projection has ~136k rows, two orders of magnitude more than the
# `layer_out-21` one. Fitting the surface and projecting every point at that size
# is needlessly slow for a geometry that is already well determined by a few
# thousand points, so take an evenly spread random subsample. Set to None to use
# every row.
SAMPLE_ROWS = 8000
SAMPLE_RANDOM_STATE = 0

df = pd.read_csv(DATA_PATH).rename(columns=COLUMN_RENAMES)
df = df[~df["time_horizon_months"].isna()]
if "log10_time_horizon_months" not in df:
    df["log10_time_horizon_months"] = df["time_horizon_months"].apply(np.log10)
df["horizon_class"] = horizon_class(df["time_horizon_months"])
df["horizon_class_label"] = horizon_class_label(df["time_horizon_months"])

df = df[~df["task"].isin(['write a one-sentence email reply', 'write a short story'])]

print("rows available:", len(df))
if SAMPLE_ROWS is not None and len(df) > SAMPLE_ROWS:
    df = df.sample(SAMPLE_ROWS, random_state=SAMPLE_RANDOM_STATE)
df = df.sort_index().reset_index(drop=True)
print("rows used:", len(df), "from", sorted(df["source_folder"].unique()))

# Shared colour settings so every scatter below uses the same class -> colour map.
CLASS_COLOR = "horizon_class_label"
CLASS_ORDER = {CLASS_COLOR: list(HORIZON_CLASS_LABELS)}
CLASS_PALETTE = px.colors.sequential.Viridis

for index, label in enumerate(HORIZON_CLASS_LABELS):
    print(index, label)
print("
out of range (< 1 second or missing):", int((df["horizon_class"] < 0).sum()))
df.groupby(["horizon_class", "horizon_class_label"]).size().rename("count").reset_index()


## Spline

In [ ]:
# The curve the surface is extruded from: a spline fitted in the PLS1-PLS2
# plane. The layer-18 artifact does not exist until this cell writes it, so the
# first run fits it even with the flag left at False.
#
# FIT_PLANE_SPLINE = True refits it from the current data and overwrites the
# artifact; False loads it (it is also fitted automatically if the file is
# missing). The extrusion only ever uses the curve's PLS1/PLS2 coordinates --
# PLS3 comes from the extrusion parameter u -- so a planar curve is all it needs.
from utils.spline_fitting import load_or_fit_plane_spline

FIT_PLANE_SPLINE = False

SPLINE_PATH = (
    REPO_ROOT / "models" / "layer18-27_pos-1_activation_curve_PLS1-PLS2-by-t_spline.joblib"
)
ENDPOINT_QUANTILE = 0.05        # tail fraction averaged into each curve endpoint
REFINEMENT_ITERATIONS = 4       # geometric re-parameterisation passes

spline, spline_metadata = load_or_fit_plane_spline(
    SPLINE_PATH,
    df[["PLS1", "PLS2"]].to_numpy(float),
    fit=FIT_PLANE_SPLINE,
    # Orient t so it increases with the time horizon rather than arbitrarily.
    order_by=df["log10_time_horizon_months"].to_numpy(float),
    endpoint_quantile=ENDPOINT_QUANTILE,
    refinement_iterations=REFINEMENT_ITERATIONS,
    coordinate_features=("PLS1", "PLS2", "PLS3"),
    metadata={
        "source_data": "data/all_datasets_layer18_27_pos-1_pls6.csv",
        "fitted_in": "PLS1-PLS2 plane (PLS3 held at 0)",
        "oriented_by": "log10_time_horizon_months",
    },
)

print("refit" if FIT_PLANE_SPLINE else "loaded", "->", spline.algorithm,
      spline.parameter_feature, "->", spline.coordinate_features)
print("t range:", np.round(spline.training_parameter_bounds, 3))


### PLS cloud with the fitted curve

In [ ]:
fig = px.scatter_3d(
    df, x="PLS1", y="PLS2", z="PLS3",
    color=CLASS_COLOR,
    category_orders=CLASS_ORDER,
    color_discrete_sequence=CLASS_PALETTE,
)
fig.update_traces(marker={"size":3})

# Overlay the fitted spline, sampled across its training range in t.
t_lo, t_hi = spline.training_parameter_bounds
t_grid = np.linspace(t_lo, t_hi, 400)
curve_xyz = spline.predict(t_grid)
fig.add_scatter3d(
    x=curve_xyz[:, 0],
    y=curve_xyz[:, 1],
    z=curve_xyz[:, 2],
    mode="lines",
    line={"width": 8, "color": "black"},
    name=f"{spline.algorithm} curve",
    customdata=t_grid,
    hovertemplate="t=%{customdata:.3f}<br>PLS1=%{x:.2f}<br>PLS2=%{y:.2f}<br>PLS3=%{z:.2f}<extra></extra>",
)
fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
fig.show()


## Extruded surface

In [ ]:
# Extrusion along PLS3: S(t, u) = (PLS1(t) + p0(u), PLS2(t) + p1(u), u), fitted
# to minimise geometric RMSE. The fitting logic lives in `vendor/utils/`.
#
# OFFSET_DEGREE 2 is the cross-section order carried over from the layer_out-21
# fit, where degree 3 measured only marginally better. The layer-18 cloud has a
# different scale, so the absolute RMSE below is not comparable to that run.
#
# Before fitting, the parameter range is extended (via the spline's tangent
# extrapolation) until the curve spans the whole data range, so no point projects
# onto a clamped endpoint.
#
# Set FIT = True to refit and overwrite the saved artifact; otherwise the saved
# model is loaded (it is also refitted automatically if no artifact exists yet).
FIT = False

from utils.surface_fitting import fit_extruded_surface, load_or_fit_extruded_surface

OFFSET_DEGREE = 2       # polynomial order of the cross-section profile p(u)

SURFACE_PATH = (
    REPO_ROOT / "models"
    / f"layer18-27_pos-1_activation_surface_PLS1-PLS2-by-t_extruded-PLS3_degree-{OFFSET_DEGREE}.joblib"
)

P = df[["PLS1", "PLS2", "PLS3"]].to_numpy(float)

surface, surface_metadata = load_or_fit_extruded_surface(
    SURFACE_PATH,
    P,
    spline,
    fit=FIT,
    offset_degree=OFFSET_DEGREE,
    extrusion_feature="PLS3",
    extend_bounds=True,     # widen t until the curve covers the data
    extension_step=0.05,    # fraction of the spline's own t span per step
    extension_margin=0.01,  # extra slack once every point is interior
    metadata={
        "source_curve": SPLINE_PATH.name,
        "source_data": "data/all_datasets_layer18_27_pos-1_pls6.csv",
        "fit_objective": "geometric RMSE (alternating least squares)",
    },
)
print("artifact:", SURFACE_PATH.name, "(refit)" if FIT else "(loaded)")
pprint(surface.metrics)
print("offset coefficients [u**0 ... u**%d] x [PLS1, PLS2]:\n" % OFFSET_DEGREE,
      surface.offset_coefficients)
print("spline t range:", np.round(spline.training_parameter_bounds, 3),
      "-> surface t range:", np.round(surface.training_parameter_bounds, 3))

# Plot the fitted surface over the points, across the extended t range.
u_lo, u_hi = P[:, 2].min(), P[:, 2].max()
pad = 0.05 * (u_hi - u_lo)
u_grid = np.linspace(u_lo - pad, u_hi + pad, 60)
t_surface = np.linspace(*surface.training_parameter_bounds, 400)
Xq, Yq, Zq = surface.grid(t_surface, u_grid)
Tq = np.broadcast_to(t_surface[:, None], Xq.shape)

fig_q = px.scatter_3d(
    df, x="PLS1", y="PLS2", z="PLS3",
    color=CLASS_COLOR,
    category_orders=CLASS_ORDER,
    color_discrete_sequence=CLASS_PALETTE,
)
fig_q.update_traces(marker={"size": 3})
fig_q.add_surface(
    x=Xq, y=Yq, z=Zq,
    surfacecolor=Tq,
    colorscale="Viridis",
    opacity=0.45,
    showscale=False,
    name="quadratic extrusion (u = PLS3)",
    showlegend=True,
    hovertemplate="t=%{surfacecolor:.3f}<br>u=PLS3=%{z:.2f}<br>PLS1=%{x:.2f}<br>PLS2=%{y:.2f}<extra></extra>",
)
fig_q.update_layout(autosize=True, width=None, height=None)
fig_q.show()


## Surface coordinates

In [ ]:
# Per-point surface coordinates (t, u) for every point in the plot above.
t_point, u_point, dist_point = surface.project(P)

coords = df.copy()
coords["t"] = t_point
coords["u"] = u_point                     # u is exactly the PLS3 coordinate
coords["surface_distance"] = dist_point
coords[["PLS1_fit", "PLS2_fit", "PLS3_fit"]] = surface.predict(t_point, u_point)

print("RMSE:", np.sqrt((dist_point**2).mean()).round(4), "  max:", dist_point.max().round(4))

### Points in surface coordinates

In [ ]:
fig = px.scatter_3d(
    coords, x="t", y="u", z="log10_time_horizon_months",
    color=CLASS_COLOR,
    category_orders=CLASS_ORDER,
    color_discrete_sequence=CLASS_PALETTE,
)
fig.update_traces(marker={"size": 3})

## Ordinal regression

In [ ]:
# Ordinal regression on the surface coordinates plus the residual principal
# components: (t, u, residual PC1-3) -> horizon_class.
# The classes are ordered, so the models below predict a rank rather than an
# unordered label; each algorithm gets its own cell. Shared knobs live here.
from utils.ordinal_regression import ORDINAL_ALGORITHMS, fit_ordinal_model

# Principal components of the reconstruction residual, shared by both models.
RESIDUAL_FEATURES = [
    "reconstruction_residual_PC1",
    "reconstruction_residual_PC2",
    "reconstruction_residual_PC3",
]

FEATURES = ["t", "u"] + RESIDUAL_FEATURES

DEGREE = 3                  # polynomial order in the input features
INTERACTION_ONLY = False    # drop pure powers, keep cross terms only
INCLUDE_BIAS = False        # every estimator carries its own intercept/cutpoints
STANDARDIZE = True          # scale the expanded features before fitting

SHARED = {
    "degree": DEGREE,
    "interaction_only": INTERACTION_ONLY,
    "include_bias": INCLUDE_BIAS,
    "standardize": STANDARDIZE,
}

X = coords[FEATURES].to_numpy(float)
y = coords["horizon_class"].to_numpy(int)
print("algorithms:", ORDINAL_ALGORITHMS)


def report(name, model, metrics):
    """Print the metrics, store the predictions on `coords`, return the crosstab."""

    pprint(metrics)
    predicted = model.predict(X)
    coords[f"horizon_class_{name}"] = predicted
    coords[f"horizon_class_label_{name}"] = np.array(HORIZON_CLASS_LABELS, dtype=object)[
        predicted
    ]
    return pd.crosstab(y, predicted, rownames=["true"], colnames=["predicted"])


### Binary decomposition on (t, u, residual PC1-3)

In [ ]:
# Binary decomposition (Frank and Hall): K-1 logistic models answering
# "is the class greater than k?", recombined into class probabilities.
BD_C = 1.0                  # inverse regularisation strength of each binary model
BD_MAX_ITER = 1000
BD_RANDOM_STATE = 0

bd_model, bd_metrics = fit_ordinal_model(
    X, y,
    algorithm="binary_decomposition",
    C=BD_C,
    max_iter=BD_MAX_ITER,
    random_state=BD_RANDOM_STATE,
    **SHARED,
)
report("binary_decomposition", bd_model, bd_metrics)


### Save the fitted classifier

In [ ]:
# Save the fitted classifier so `predict_horizon_batch_prompts.ipynb` can load it.
# The surface and the spline are already saved by their own cells; this is the
# last piece needed to predict without refitting anything.
from utils.ordinal_regression import save_ordinal_model

SAVE_CLASSIFIER = True

CLASSIFIER_PATH = (
    REPO_ROOT / "models"
    / f"layer18-27_pos-1_horizon_class_ordinal_binary-decomposition_degree-{DEGREE}.joblib"
)

if SAVE_CLASSIFIER:
    saved = save_ordinal_model(
        bd_model,
        CLASSIFIER_PATH,
        FEATURES,
        metadata={
            "algorithm": "binary_decomposition",
            "surface_artifact": SURFACE_PATH.name,
            "source_data": "data/all_datasets_layer18_27_pos-1_pls6.csv",
            "horizon_class_labels": list(HORIZON_CLASS_LABELS),
            "knobs": {
                "degree": DEGREE,
                "interaction_only": INTERACTION_ONLY,
                "include_bias": INCLUDE_BIAS,
                "standardize": STANDARDIZE,
                "C": BD_C,
                "max_iter": BD_MAX_ITER,
                "random_state": BD_RANDOM_STATE,
            },
            "training_metrics": bd_metrics,
        },
    )
    print("saved:", saved)
else:
    print("not saved; set SAVE_CLASSIFIER = True to write", CLASSIFIER_PATH.name)


### Baseline on (PLS1-3, residual PC1-3)

In [ ]:
# Same ordinal model, but straight from the PLS coordinates instead of the
# surface coordinates: (PLS1-3, residual PC1-3) -> horizon_class. This is the
# baseline the (t, u) parameterisation has to beat -- it sees the raw 3D
# position, while (t, u) sees only where a point sits on the fitted surface.
# Both models get the same residual principal components.
# This projection carries six PLS components, and the three the surface ignores
# hold a lot of the class signal: on an 8,000-row sample the baseline goes from
# 0.644 accuracy on PLS1-3 to 0.793 on PLS1-6. Set PLS_COMPONENTS = 3 for the
# like-for-like comparison against the surface, which only ever sees PLS1-3.
PLS_COMPONENTS = 6
PLS_FEATURES = [f"PLS{i}" for i in range(1, PLS_COMPONENTS + 1)] + RESIDUAL_FEATURES
PLS_ALGORITHM = "binary_decomposition"   # any name from ORDINAL_ALGORITHMS
PLS_OPTIONS = {"C": BD_C, "max_iter": BD_MAX_ITER, "random_state": BD_RANDOM_STATE}

X_pls = coords[PLS_FEATURES].to_numpy(float)

pls_model, pls_metrics = fit_ordinal_model(
    X_pls, y,
    algorithm=PLS_ALGORITHM,
    **PLS_OPTIONS,
    **SHARED,
)
pprint(pls_metrics)

pls_predicted = pls_model.predict(X_pls)
coords["horizon_class_pls"] = pls_predicted
coords["horizon_class_label_pls"] = np.array(HORIZON_CLASS_LABELS, dtype=object)[
    pls_predicted
]

print(f"\n(t, u) vs PLS1-{PLS_COMPONENTS}:")
display(
    pd.DataFrame([bd_metrics | {"features": "t, u"}, pls_metrics | {"features": f"PLS1-{PLS_COMPONENTS}"}])
    .set_index("features")
)
pd.crosstab(y, pls_predicted, rownames=["true"], colnames=["predicted"])
